# LeetCode 3348 — Smallest Divisible Digit Product II bằng Python

- Link: https://leetcode.com/problems/smallest-divisible-digit-product-ii/
- Độ khó: **Hard**
- Chủ đề: **Tham lam xây số nhỏ nhất ≥ num** (giữ tiền tố dài nhất, tăng một chữ số, điền hậu tố nhỏ nhất), **phân tích thừa số 2·3·5·7**, đếm **số chữ số tối thiểu** để phủ một yêu cầu

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def smallestNumber(self, num: str, t: int) -> str:
```

## 1. Mục tiêu học

1. Nhận ra tích chữ số chỉ chứa thừa số nguyên tố `2, 3, 5, 7` ⇒ nếu `t` có thừa số khác thì **không bao giờ** có đáp án (`"-1"`); ngược lại **luôn** có đáp án (có thể dài hơn `num`).
2. Biểu diễn "yêu cầu còn thiếu" bằng **vector số mũ** `(a, b, c, d)` của `2, 3, 5, 7`.
3. Tính **số chữ số tối thiểu** để phủ một yêu cầu và **hậu tố nhỏ nhất** có độ dài cho trước (tham lam 9, 8, 7, … rồi sắp tăng, đệm `1` ở đầu).
4. Khung "số nhỏ nhất `≥ num`": thử giữ tiền tố `num[:i]` **dài nhất có thể**, đặt chữ số `> num[i]` tại `i`, phần sau điền nhỏ nhất.
5. Xử lý chữ số `0` trong `num` (tiền tố chứa `0` không thể giữ).

## 2. Đề bài & ví dụ

Cho chuỗi `num` (số nguyên dương) và `t`. Một số là **zero-free** nếu không có chữ số `0`. Trả về **số zero-free nhỏ nhất `≥ num`** có **tích các chữ số chia hết cho `t`**; nếu không tồn tại, trả `"-1"`.

**Ví dụ 1:** `num = "1234"`, `t = 256` → `"1488"` (tích `256`).
**Ví dụ 2:** `num = "12355"`, `t = 50` → `"12355"` (tích `150`).
**Ví dụ 3:** `num = "11111"`, `t = 26` → `"-1"`.

**Ràng buộc:** `2 ≤ len(num) ≤ 2·10^5`; `num` không có số 0 đầu; `1 ≤ t ≤ 10^14`.

In [1]:
from typing import List
from math import gcd, prod
from functools import lru_cache

examples = [("1234", 256, "1488"), ("12355", 50, "12355"), ("11111", 26, "-1")]
for num, t, exp in examples:
    print(f"num = {num}, t = {t} -> kỳ vọng {exp}")

num = 1234, t = 256 -> kỳ vọng 1488
num = 12355, t = 50 -> kỳ vọng 12355
num = 11111, t = 26 -> kỳ vọng -1


## 3. Kiến thức nền

### 3.1. Phân tích `t` theo `2, 3, 5, 7`

Mỗi chữ số `1..9` chỉ có thừa số nguyên tố trong `{2, 3, 5, 7}`. Nếu `t` còn thừa số khác (ví dụ `13` trong `26`) thì không có tích chữ số nào chia hết cho `t`.

In [2]:
PR = [2, 3, 5, 7]
DIGIT_EXP = {d: [0] * 4 for d in range(1, 10)}
for d in range(2, 10):
    x = d
    for k, p in enumerate(PR):
        while x % p == 0:
            DIGIT_EXP[d][k] += 1
            x //= p

def factor(t):
    e = [0] * 4
    for k, p in enumerate(PR):
        while t % p == 0:
            e[k] += 1
            t //= p
    return e if t == 1 else None

print({d: DIGIT_EXP[d] for d in range(1, 10)})
print('256 ->', factor(256), '| 50 ->', factor(50), '| 26 ->', factor(26))

{1: [0, 0, 0, 0], 2: [1, 0, 0, 0], 3: [0, 1, 0, 0], 4: [2, 0, 0, 0], 5: [0, 0, 1, 0], 6: [1, 1, 0, 0], 7: [0, 0, 0, 1], 8: [3, 0, 0, 0], 9: [0, 2, 0, 0]}
256 -> [8, 0, 0, 0] | 50 -> [1, 0, 2, 0] | 26 -> None


### 3.2. Trừ yêu cầu khi dùng một chữ số

Dùng chữ số `d` làm giảm yêu cầu: `r_k ← max(0, r_k − e_k(d))`.

In [3]:
def sub(r, d):
    e = DIGIT_EXP[d]
    return [max(0, r[k] - e[k]) for k in range(4)]

print(sub([8, 0, 0, 0], 8), sub([1, 1, 0, 0], 6), sub([0, 2, 1, 0], 9))

[5, 0, 0, 0] [0, 0, 0, 0] [0, 0, 1, 0]


### 3.3. Số chữ số tối thiểu & hậu tố nhỏ nhất

Với yêu cầu `r = (a, b, c, d)`:

- Mỗi `7` và mỗi `5` cần một chữ số riêng (`7`, `5`).
- Thừa số `3`: gom đôi thành `9`; thừa số `2`: gom ba thành `8`.
- Phần dư (`a mod 3 ∈ {0,1,2}`, `b mod 2 ∈ {0,1}`): `2·3 → 6`, `2² → 4`, `2 → 2`, `3 → 3`.

Đây là cách dùng **ít chữ số nhất**, và sắp các chữ số **tăng dần** (đệm `1` phía trước) cho **hậu tố nhỏ nhất** với độ dài cho trước — ta kiểm chứng bằng vét cạn ngay dưới.

In [4]:
def greedy_digits(r):
    a, b, c, e = r
    ds = [7] * e + [5] * c + [9] * (b // 2) + [8] * (a // 3)
    a, b = a % 3, b % 2
    if b and a:
        ds.append(6); a -= 1; b = 0
    if a == 2:
        ds.append(4)
    elif a == 1:
        ds.append(2)
    if b:
        ds.append(3)
    return sorted(ds)

def min_digits(r):
    return len(greedy_digits(r))

def smallest_suffix(r, L):
    ds = greedy_digits(r)
    if len(ds) > L:
        return None
    return '1' * (L - len(ds)) + ''.join(map(str, ds))

import itertools
for t in range(1, 300):
    r = factor(t)
    if r is None:
        continue
    for L in range(1, 4):
        best = next((''.join(map(str, tup)) for tup in itertools.product(range(1, 10), repeat=L)
                     if prod(tup) % t == 0), None)
        assert smallest_suffix(r, L) == best, (t, L)
print('hậu tố tham lam = hậu tố nhỏ nhất (vét cạn t < 300, L <= 3). Ví dụ t = 256:', greedy_digits(factor(256)))

hậu tố tham lam = hậu tố nhỏ nhất (vét cạn t < 300, L <= 3). Ví dụ t = 256: [4, 8, 8]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Số nhỏ nhất `≥ num` có cùng độ dài `n` có dạng: **giữ nguyên** `num[:i]`, đặt chữ số `x > num[i]` tại vị trí `i`, rồi phần sau **tùy ý** (dài `n − i − 1`). Muốn số nhỏ nhất ⇒ ưu tiên `i` **lớn nhất** (tiền tố chung dài nhất), rồi `x` nhỏ nhất, rồi hậu tố nhỏ nhất.

Ví dụ 1: `num = 1234`, `t = 256 = 2^8`.

- Giữ cả `1234`? tích `24 = 2^3·3` — không chia hết.
- `i = 3`: giữ `123`, yêu cầu còn `2^7`, đặt `5..9` vào cuối — `8` chỉ cho `2^3`, không đủ (cần `2^7` với 1 chữ số).
- `i = 2`: giữ `12` (còn `2^7`), đặt `x ∈ {4..9}` rồi hậu tố 1 chữ số: `x = 8` còn `2^4` — 1 chữ số không đủ…
- `i = 1`: giữ `1`, đặt `x ≥ 3`: `x = 4` còn `2^6` ⇒ cần `88` (2 chữ số) ✓ ⇒ `1` `4` `88` = `1488` ✓.

## 5. Cách 1 — Brute Force: tăng dần từ `num`

In [5]:
def brute(num, t):
    if factor(t) is None:
        return "-1"
    x = int(num)
    while True:
        s = str(x)
        if '0' not in s and prod(map(int, s)) % t == 0:
            return s
        x += 1

for num, t, exp in examples:
    print(brute(num, t), '| kỳ vọng', exp)

1488 | kỳ vọng 1488
12355 | kỳ vọng 12355
-1 | kỳ vọng -1


**Độ phức tạp:** không bị chặn tốt (có thể phải tăng rất nhiều lần) — chỉ dùng cho số nhỏ.

## 6. Ý tưởng thuật toán / bước đột phá

1. `r0 = factor(t)`; nếu `None` ⇒ `"-1"`.
2. Tính `pre[i]` = yêu cầu còn lại sau khi giữ `num[:i]`, chỉ tới **chữ số 0 đầu tiên** `z` (tiền tố chứa 0 không giữ được).
3. Nếu `num` không có 0 và `pre[n] = 0` ⇒ trả `num`.
4. Với `i = min(z, n − 1)` giảm về `0`: với `x = num[i] + 1 .. 9`: nếu `min_digits(sub(pre[i], x)) ≤ n − i − 1` ⇒ đáp án `num[:i] + x + smallest_suffix(...)`.
5. Không có đáp án cùng độ dài ⇒ số dài `L = max(n + 1, min_digits(r0))`: `smallest_suffix(r0, L)`.

Mỗi lần thử là `O(1)` (vector 4 chiều), dựng đáp án một lần `O(n)` ⇒ tổng `O(9n)`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Cùng khung, nhưng yêu cầu là **số nguyên** và số chữ số tối thiểu tính bằng **DP có nhớ**

Không cần công thức đóng: `f(r)` = số chữ số tối thiểu để tích chia hết cho `r`; `f(1) = 0`, `f(r) = 1 + min_x f(r / gcd(r, x))`. Các `r` đều là ước của `t` nên số trạng thái nhỏ. Hậu tố nhỏ nhất dựng **từng chữ số**: chọn `x` nhỏ nhất mà phần còn lại vẫn kịp phủ.

In [6]:
def dp_solution(num, t):
    if factor(t) is None:
        return "-1"

    @lru_cache(maxsize=None)
    def f(r):
        if r == 1:
            return 0
        return 1 + min(f(r // gcd(r, x)) for x in range(2, 10) if gcd(r, x) > 1)

    def build(r, L):
        out = []
        for pos in range(L):
            for x in range(1, 10):
                nr = r // gcd(r, x)
                if f(nr) <= L - pos - 1:
                    out.append(str(x)); r = nr
                    break
        return ''.join(out)

    n = len(num)
    pre = [t]
    z = n
    for i, ch in enumerate(num):
        if ch == '0':
            z = i
            break
        pre.append(pre[-1] // gcd(pre[-1], int(ch)))
    if z == n and pre[n] == 1:
        return num
    for i in range(min(z, n - 1), -1, -1):
        L = n - i - 1
        for x in range(int(num[i]) + 1, 10):
            r = pre[i] // gcd(pre[i], x)
            if f(r) <= L:
                return num[:i] + str(x) + build(r, L)
    L = max(n + 1, f(t))
    return build(t, L)

for num, t, exp in examples:
    print(dp_solution(num, t), '| kỳ vọng', exp)

1488 | kỳ vọng 1488
12355 | kỳ vọng 12355
-1 | kỳ vọng -1


### 7.2. Cách 3 — Vector số mũ + công thức đóng (lời giải chính)

In [7]:
def exp_solution(num, t):
    r0 = factor(t)
    if r0 is None:
        return "-1"
    n = len(num)
    pre = [r0]
    z = n
    for i, ch in enumerate(num):
        if ch == '0':
            z = i
            break
        pre.append(sub(pre[-1], int(ch)))
    if z == n and pre[n] == [0, 0, 0, 0]:
        return num
    for i in range(min(z, n - 1), -1, -1):
        L = n - i - 1
        for x in range(int(num[i]) + 1, 10):
            r = sub(pre[i], x)
            if min_digits(r) <= L:
                return num[:i] + str(x) + smallest_suffix(r, L)
    L = max(n + 1, min_digits(r0))
    return smallest_suffix(r0, L)

for num, t, exp in examples:
    print(exp_solution(num, t), '| kỳ vọng', exp)

1488 | kỳ vọng 1488
12355 | kỳ vọng 12355
-1 | kỳ vọng -1


### 7.3. Vì sao ưu tiên `i` lớn nhất là đúng

Hai ứng viên cùng độ dài `n`: ứng viên giữ tiền tố `num[:i]` và tăng tại `i` so với ứng viên giữ `num[:j]` (`j < i`) và tăng tại `j`: tại vị trí `j`, ứng viên thứ nhất có `num[j]`, thứ hai có chữ số `> num[j]` ⇒ ứng viên thứ nhất nhỏ hơn. Mọi số cùng độ dài luôn nhỏ hơn số dài hơn ⇒ chỉ chuyển sang độ dài `n + 1` khi không còn lựa chọn.

## 8. TRACE từng bước

Trace cách 3 trên `num = "1234"`, `t = 256`.

In [8]:
def trace(num, t):
    r0 = factor(t)
    n = len(num)
    pre = [r0]
    for ch in num:
        if ch == '0':
            break
        pre.append(sub(pre[-1], int(ch)))
    print('yêu cầu (mũ 2,3,5,7) sau mỗi tiền tố:', pre)
    for i in range(n - 1, -1, -1):
        L = n - i - 1
        for x in range(int(num[i]) + 1, 10):
            r = sub(pre[i], x)
            need = min_digits(r)
            mark = 'OK' if need <= L else ''
            print(f"  i = {i}, giữ {num[:i]!r}, đặt {x}: còn {r}, cần {need} chữ số, có {L} {mark}")
            if need <= L:
                ans = num[:i] + str(x) + smallest_suffix(r, L)
                print('đáp án:', ans, '| vét cạn:', brute(num, t))
                return

trace("1234", 256)

yêu cầu (mũ 2,3,5,7) sau mỗi tiền tố: [[8, 0, 0, 0], [8, 0, 0, 0], [7, 0, 0, 0], [7, 0, 0, 0], [5, 0, 0, 0]]
  i = 3, giữ '123', đặt 5: còn [7, 0, 0, 0], cần 3 chữ số, có 0 
  i = 3, giữ '123', đặt 6: còn [6, 0, 0, 0], cần 2 chữ số, có 0 
  i = 3, giữ '123', đặt 7: còn [7, 0, 0, 0], cần 3 chữ số, có 0 
  i = 3, giữ '123', đặt 8: còn [4, 0, 0, 0], cần 2 chữ số, có 0 
  i = 3, giữ '123', đặt 9: còn [7, 0, 0, 0], cần 3 chữ số, có 0 
  i = 2, giữ '12', đặt 4: còn [5, 0, 0, 0], cần 2 chữ số, có 1 
  i = 2, giữ '12', đặt 5: còn [7, 0, 0, 0], cần 3 chữ số, có 1 
  i = 2, giữ '12', đặt 6: còn [6, 0, 0, 0], cần 2 chữ số, có 1 
  i = 2, giữ '12', đặt 7: còn [7, 0, 0, 0], cần 3 chữ số, có 1 
  i = 2, giữ '12', đặt 8: còn [4, 0, 0, 0], cần 2 chữ số, có 1 
  i = 2, giữ '12', đặt 9: còn [7, 0, 0, 0], cần 3 chữ số, có 1 
  i = 1, giữ '1', đặt 3: còn [8, 0, 0, 0], cần 3 chữ số, có 2 
  i = 1, giữ '1', đặt 4: còn [6, 0, 0, 0], cần 2 chữ số, có 2 OK
đáp án: 1488 | vét cạn: 1488


## 9. Code nộp LeetCode

In [9]:
class Solution:
    def smallestNumber(self, num: str, t: int) -> str:
        P = (2, 3, 5, 7)
        E = [[0, 0, 0, 0] for _ in range(10)]
        for d in range(2, 10):
            x = d
            for k, p in enumerate(P):
                while x % p == 0:
                    E[d][k] += 1
                    x //= p
        r0 = [0, 0, 0, 0]
        for k, p in enumerate(P):
            while t % p == 0:
                r0[k] += 1
                t //= p
        if t != 1:
            return "-1"

        def digits_for(r):
            a, b, c, e = r
            ds = [7] * e + [5] * c + [9] * (b // 2) + [8] * (a // 3)
            a, b = a % 3, b % 2
            if b and a:
                ds.append(6); a -= 1; b = 0
            if a == 2:
                ds.append(4)
            elif a == 1:
                ds.append(2)
            if b:
                ds.append(3)
            ds.sort()
            return ds

        def need(r):
            a, b, c, e = r
            cnt = e + c + b // 2 + a // 3
            a, b = a % 3, b % 2
            if a and b:
                return cnt + 1 + (a == 2)
            return cnt + (a > 0) + b

        n = len(num)
        pre = [r0]
        z = n
        for i, ch in enumerate(num):
            if ch == '0':
                z = i
                break
            e, r = E[int(ch)], pre[-1]
            pre.append([max(0, r[0] - e[0]), max(0, r[1] - e[1]), max(0, r[2] - e[2]), max(0, r[3] - e[3])])
        if z == n and not any(pre[n]):
            return num
        for i in range(min(z, n - 1), -1, -1):
            L = n - i - 1
            r = pre[i]
            for x in range(int(num[i]) + 1, 10):
                e = E[x]
                nr = [max(0, r[0] - e[0]), max(0, r[1] - e[1]), max(0, r[2] - e[2]), max(0, r[3] - e[3])]
                if need(nr) <= L:
                    ds = digits_for(nr)
                    return num[:i] + str(x) + '1' * (L - len(ds)) + ''.join(map(str, ds))
        ds = digits_for(r0)
        L = max(n + 1, len(ds))
        return '1' * (L - len(ds)) + ''.join(map(str, ds))

for num, t, exp in examples:
    got = Solution().smallestNumber(num, t)
    print(got, 'OK' if got == exp else 'SAI')

1488 OK
12355 OK
-1 OK


## 10. Vì sao thuật toán đúng

1. **Tồn tại:** nếu `t` chỉ có thừa số `2, 3, 5, 7` thì số `1…1` + `greedy_digits(r0)` đủ dài (≥ `n + 1`) luôn hợp lệ và `> num` ⇒ luôn có đáp án; ngược lại không tích chữ số nào chia hết cho `t`.
2. **Thứ tự xét:** (mục 7.3) đáp án cùng độ dài luôn tốt hơn dài hơn; trong cùng độ dài, tiền tố chung dài hơn tốt hơn; cùng `i`, chữ số `x` nhỏ hơn tốt hơn; cùng `x`, hậu tố nhỏ nhất.
3. **Khả thi của một lựa chọn** `(i, x)` ⇔ yêu cầu còn lại phủ được bằng `L` chữ số ⇔ `min_digits(r) ≤ L` (thêm chữ số `1` không ảnh hưởng tích).
4. **Hậu tố nhỏ nhất:** dùng ít chữ số "có ích" nhất ⇒ nhiều số `1` đứng đầu nhất; trong số các bộ ít chữ số nhất, bộ tham lam (gom `9`, `8`, rồi `6/4/3/2`) cho chữ số **nhỏ nhất ở từng vị trí** sau khi sắp tăng — kiểm chứng vét cạn ở mục 3.3 và mục 12.
5. **Chữ số 0:** tiền tố chứa `0` không zero-free nên chỉ xét `i ≤ z`; tại `i = z` ta thay `0` bằng `x ≥ 1`.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Tăng dần | không bị chặn tốt | `O(n)` |
| 2. DP có nhớ trên ước của `t` | `O(9n + D·9)` (`D` = số ước dùng tới) | `O(n + D)` |
| 3. Vector số mũ (nộp) | `O(9n)` | `O(n)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [10]:
impls = {'dp_solution': dp_solution, 'exp_solution': exp_solution,
         'submit': lambda a, b: Solution().smallestNumber(a, b)}
edge = [
    ("10", 1, "11"),          # 0 phải bị thay
    ("99", 7, "117"),         # phải dài thêm
    ("99", 4, "114"),
    ("11", 1, "11"),          # t = 1: chính num (zero-free)
    ("20", 5, "25"),
    ("19", 13, "-1"),
    ("999", 2 ** 10, "2888"), # 2^10 cần 4 chữ số: 2,8,8,8
]
for num, t, exp in examples + edge:
    assert brute(num, t) == exp, (num, t, brute(num, t))
    for name, f in impls.items():
        assert f(num, t) == exp, (name, num, t, f(num, t))
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với vét cạn

In [11]:
import random
rng = random.Random(3348)
cnt = 0
for _ in range(1200):
    num = str(rng.randint(10, 99999))
    if rng.random() < 0.3:
        num = num[:1] + ''.join(rng.choice('0123456789') for _ in range(len(num) - 1))
    t = rng.choice([rng.randint(1, 300),
                    2 ** rng.randint(0, 9) * 3 ** rng.randint(0, 4),
                    5 ** rng.randint(0, 2) * 7 ** rng.randint(0, 2)])
    exp = brute(num, t)
    for name, f in impls.items():
        assert f(num, t) == exp, (name, num, t)
    cnt += 1
print('Khớp vét cạn trên', cnt, 'bộ (num, t).')

Khớp vét cạn trên 1200 bộ (num, t).


### 12.3. Ca lớn `len(num) = 2·10^5`

In [12]:
import time
N = 2 * 10 ** 5
cases = {
    'ngẫu nhiên, t = 2^40·3^5': (''.join(rng.choice('123456789') for _ in range(N)), 2 ** 40 * 3 ** 5),
    'toàn 9, t = 7^16': ('9' * N, 7 ** 16),
    'có 0 ở giữa, t = 10^14': ('1' * (N // 2) + '0' + '1' * (N // 2 - 1), 10 ** 14),
    'toàn 1, t = 1': ('1' * N, 1),
}
for name, (num, t) in cases.items():
    tt = time.time(); a = Solution().smallestNumber(num, t); t1 = time.time() - tt
    tt = time.time(); b = dp_solution(num, t); t2 = time.time() - tt
    assert a == b and len(a) >= N and '0' not in a and (len(a), a) >= (len(num), num)
    print(f"{name:28s}: độ dài {len(a)} | đuôi ...{a[-12:]} | nộp {t1:.2f}s | dp {t2:.2f}s")

ngẫu nhiên, t = 2^40·3^5    : độ dài 200000 | đuôi ...522452496775 | nộp 0.12s | dp 0.03s


toàn 9, t = 7^16            : độ dài 200001 | đuôi ...777777777777 | nộp 0.11s | dp 0.12s
có 0 ở giữa, t = 10^14      : độ dài 200000 | đuôi ...555555558888 | nộp 0.07s | dp 0.07s


toàn 1, t = 1               : độ dài 200000 | đuôi ...111111111111 | nộp 0.13s | dp 0.03s


Kiểm tra độc lập cho ca lớn: tích chữ số của đáp án chia hết cho `t` (tính bằng vector số mũ để tránh số khổng lồ).

In [13]:
def divisible(ans, t):
    r = factor(t)
    have = [0] * 4
    for ch in ans:
        for k in range(4):
            have[k] += DIGIT_EXP[int(ch)][k]
    return all(have[k] >= r[k] for k in range(4))

for name, (num, t) in cases.items():
    assert divisible(Solution().smallestNumber(num, t), t)
print('Mọi đáp án ca lớn đều có tích chữ số chia hết cho t.')

Mọi đáp án ca lớn đều có tích chữ số chia hết cho t.


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Không kiểm tra thừa số nguyên tố khác `2, 3, 5, 7`

In [14]:
def wrong_no_prime_check(num, t):
    # SAI: chỉ lấy phần 2,3,5,7 của t, bỏ qua phần dư
    e = [0] * 4
    for k, p in enumerate(PR):
        while t % p == 0:
            e[k] += 1; t //= p
    return exp_solution(num, prod(p ** a for p, a in zip(PR, e)))

print('sai :', wrong_no_prime_check("11111", 26), '| đúng:', Solution().smallestNumber("11111", 26))

sai : 11112 | đúng: -1


**Vì sao sai:** `26 = 2 · 13`; số `11112` có tích `2`, không chia hết cho `13`. Không chữ số nào chứa thừa số `13`.

### Lỗi 2 — Điền hậu tố bằng chữ số nhỏ trước (`2, 2, 2` thay vì `8`)

In [15]:
def wrong_small_digits(num, t):
    r0 = factor(t)
    if r0 is None:
        return "-1"
    small = lambda r: sorted([2] * r[0] + [3] * r[1] + [5] * r[2] + [7] * r[3])   # SAI: không gom
    n = len(num)
    pre = [r0]; z = n
    for i, ch in enumerate(num):
        if ch == '0':
            z = i; break
        pre.append(sub(pre[-1], int(ch)))
    if z == n and pre[n] == [0] * 4:
        return num
    for i in range(min(z, n - 1), -1, -1):
        L = n - i - 1
        for x in range(int(num[i]) + 1, 10):
            ds = small(sub(pre[i], x))
            if len(ds) <= L:
                return num[:i] + str(x) + '1' * (L - len(ds)) + ''.join(map(str, ds))
    ds = small(r0); L = max(n + 1, len(ds))
    return '1' * (L - len(ds)) + ''.join(map(str, ds))

print('sai :', wrong_small_digits("99", 4), '| đúng:', Solution().smallestNumber("99", 4))

sai : 122 | đúng: 114


**Vì sao sai:** `4 = 2²` chỉ cần **một** chữ số `4`; dùng `2, 2` tốn hai chữ số, mất một chỗ cho số `1` ở đầu ⇒ `122 > 114`.

### Lỗi 3 — Quên điều kiện zero-free: "tích bằng 0 chia hết mọi `t`"

In [16]:
def wrong_allow_zero(num, t):
    if prod(map(int, num)) % t == 0:      # SAI: num có chữ số 0 thì tích = 0
        return num
    return Solution().smallestNumber(num, t)

print('sai :', wrong_allow_zero("105", 5), '| đúng:', Solution().smallestNumber("105", 5))

sai : 105 | đúng: 115


**Vì sao sai:** `105` chứa `0` nên không zero-free; đáp án đúng là `115` (thay `0` bằng `1`, tích `5`).

### Lỗi 4 — Chỉ tìm trong các số cùng độ dài với `num`

In [17]:
def wrong_same_length_only(num, t):
    r0 = factor(t)
    if r0 is None:
        return "-1"
    n = len(num)
    pre = [r0]; z = n
    for i, ch in enumerate(num):
        if ch == '0':
            z = i; break
        pre.append(sub(pre[-1], int(ch)))
    if z == n and pre[n] == [0] * 4:
        return num
    for i in range(min(z, n - 1), -1, -1):
        for x in range(int(num[i]) + 1, 10):
            s = smallest_suffix(sub(pre[i], x), n - i - 1)
            if s is not None:
                return num[:i] + str(x) + s
    return "-1"                          # SAI: quên đáp án dài hơn

print('sai :', wrong_same_length_only("99", 7), '| đúng:', Solution().smallestNumber("99", 7))

sai : -1 | đúng: 117


**Vì sao sai:** không số hai chữ số nào `≥ 99` có tích chia hết cho `7` (chỉ có `99`), nhưng `117` thì có — đáp án có thể dài hơn `num`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Số nhỏ nhất `≥ num` thỏa điều kiện" ⇒ thử giữ tiền tố dài nhất, tăng một chữ số, hậu tố nhỏ nhất (cần hàm **kiểm khả thi nhanh** cho hậu tố).
- Tích chữ số ⇒ nghĩ ngay tới vector số mũ của `2, 3, 5, 7`.
- Tham lam "gom thành chữ số lớn" (`9 = 3²`, `8 = 2³`) cho số chữ số ít nhất — như LC 2847.

## 15. Biến thể của bài

1. **LC 3345 (bản I):** `num ≤ 100`, `t ≤ 10` — vét cạn tăng dần.
2. **LC 2847 — Smallest Number With Given Digit Product:** tích **bằng đúng** `n`.
3. Số **lớn nhất `≤ num`** thỏa điều kiện (đối xứng, cẩn thận số ngắn hơn).
4. Điều kiện **tổng chữ số chia hết cho `t`** thay vì tích.

In [18]:
# Biến thể 2: tích bằng đúng n
def smallest_with_product(n):
    if n == 1:
        return "1"
    ds = []
    for d in range(9, 1, -1):
        while n % d == 0:
            ds.append(d); n //= d
    return ''.join(map(str, sorted(ds))) if n == 1 else "-1"

def smallest_with_product_brute(n):
    for x in range(1, 10 ** 5):
        if '0' not in str(x) and prod(map(int, str(x))) == n:
            return str(x)
    return "-1"

for n in range(1, 400):
    assert smallest_with_product(n) == smallest_with_product_brute(n), n
print('LC 2847 (tích đúng bằng n): tham lam 9..2 khớp vét cạn, ví dụ 105 ->', smallest_with_product(105))

LC 2847 (tích đúng bằng n): tham lam 9..2 khớp vét cạn, ví dụ 105 -> 357


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Tăng dần | không chặn | Oracle |
| 2 | Khung tham lam + DP trên ước `t` | `O(9n + 9D)` | Tổng quát, không cần công thức |
| 3 | Khung tham lam + công thức đóng | `O(9n)` | **Nộp bài** |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `num = "15"`, `t = 10`.

**Bài 2 (tính tay):** `num = "100"`, `t = 49`.

**Bài 3 (điền TODO):** trả về **số chữ số tối thiểu** của một số zero-free có tích chia hết cho `t` (hoặc `-1`).

**Bài 4 (tự viết lại):** số zero-free nhỏ nhất **`> num`** (lớn hơn nghiêm ngặt).

**Bài 5 (mini challenge):** biến thể 4 — tổng chữ số chia hết cho `t` (`t ≤ 100`), số nhỏ nhất `≥ num`.

In [19]:
def min_len_todo(t):
    # TODO
    raise NotImplementedError

def strictly_greater_todo(num, t):
    # TODO
    raise NotImplementedError

def digit_sum_divisible_todo(num, t):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Cần thừa số `2` và `5`. (2) `15`: tích `5`; `16..24`… (3) `25`.

**Bài 2.** (1) Thay `0`… nhưng giữ `1`, cần `7·7`. (2) `1` + `77` = `177`. (3) Đối chiếu vét cạn.

**Bài 3.** (1) `factor(t)`. (2) `min_digits`. (3) Nhớ `t = 1` ⇒ 1 chữ số.

**Bài 4.** (1) Gọi lời giải cho `num + 1` (cộng 1 trên chuỗi). (2) Cẩn thận `999…9 + 1`. (3) Chữ số 0 mới sinh ra được xử lý tự động.

**Bài 5.** (1) Cùng khung: giữ tiền tố, tăng một chữ số. (2) Hậu tố dài `L`: tổng có thể đạt mọi giá trị trong `[L, 9L]`. (3) Tham lam từng vị trí của hậu tố: chữ số nhỏ nhất mà phần còn lại vẫn đạt được số dư cần.

## 19. Đáp án tham khảo cho bài tập

In [20]:
# Bài 1, 2, 3
assert Solution().smallestNumber("15", 10) == "25" == brute("15", 10)
assert Solution().smallestNumber("100", 49) == "177" == brute("100", 49)

def min_len(t):
    r = factor(t)
    return -1 if r is None else max(1, min_digits(r))

for t in range(1, 500):
    r = factor(t)
    if r is None:
        assert min_len(t) == -1
        continue
    exp = next(L for L in range(1, 10) if any(prod(tup) % t == 0 for tup in itertools.product(range(1, 10), repeat=L)))
    assert min_len(t) == exp, t
print('Bài 1: 25 | Bài 2: 177 | Bài 3 khớp vét cạn.')

Bài 1: 25 | Bài 2: 177 | Bài 3 khớp vét cạn.


In [21]:
# Bài 4, 5
def strictly_greater(num, t):
    nxt = str(int(num) + 1)
    return Solution().smallestNumber(nxt, t)

def digit_sum_divisible(num, t):
    def can(rem, L):            # đạt tổng ≡ rem (mod t) với L chữ số 1..9 ?
        if L == 0:
            return rem % t == 0
        return any((s - rem) % t == 0 for s in range(L, 9 * L + 1)) if 9 * L - L + 1 < t else True
    def build(rem, L):
        out = []
        for pos in range(L):
            for x in range(1, 10):
                if can((rem - x) % t, L - pos - 1):
                    out.append(str(x)); rem = (rem - x) % t
                    break
        return ''.join(out)
    n = len(num)
    s = [0]; z = n
    for i, ch in enumerate(num):
        if ch == '0':
            z = i; break
        s.append(s[-1] + int(ch))
    if z == n and s[n] % t == 0:
        return num
    for i in range(min(z, n - 1), -1, -1):
        L = n - i - 1
        for x in range(int(num[i]) + 1, 10):
            rem = (-(s[i] + x)) % t
            if can(rem, L):
                return num[:i] + str(x) + build(rem, L)
    L = n + 1
    while not can(0, L):
        L += 1
    return build(0, L)

def digit_sum_brute(num, t):
    x = int(num)
    while True:
        q = str(x)
        if '0' not in q and sum(map(int, q)) % t == 0:
            return q
        x += 1

rng = random.Random(4)
for _ in range(400):
    num = str(rng.randint(10, 9999)); t = rng.randint(1, 40)
    assert digit_sum_divisible(num, t) == digit_sum_brute(num, t), (num, t)
    t2 = rng.choice([1, 2, 6, 8, 10, 49, 64])
    assert strictly_greater(num, t2) == brute(str(int(num) + 1), t2)
print('Bài 4, 5 khớp vét cạn.')

Bài 4, 5 khớp vét cạn.


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
r0 = mũ (2,3,5,7) của t; dư thừa số khác -> "-1"
pre[i] = yêu cầu sau tiền tố num[:i] (dừng ở chữ số 0 đầu tiên z)
nếu num zero-free và pre[n] == 0: return num
for i từ min(z, n-1) về 0: for x từ num[i]+1 tới 9:
    r = pre[i] - mũ(x); nếu need(r) <= n-i-1: return num[:i] + x + '1'*… + sorted(greedy(r))
return '1'*(L - need(r0)) + greedy(r0) với L = max(n+1, need(r0))
```

**Checklist:**

- [ ] Kiểm tra `t` chỉ có thừa số `2, 3, 5, 7`?
- [ ] Không giữ tiền tố chứa chữ số `0`?
- [ ] Hậu tố gom `9`, `8`, rồi `6/4/3/2`, sắp tăng, đệm `1` ở đầu?
- [ ] Có phương án dài hơn `num` khi cùng độ dài không được?
- [ ] Trường hợp `num` đã thỏa (trả về chính nó)?